# Raster calculations for Semantic segmentation modelling
Code based on given code from Gitbook > Training data > Vector to masks  

Chose to go with OSM building footprints, even though they are not as detailed and precise as the swissBUILDINGS3D dataset. Maybe post-checks bc of that ????

In [2]:
# loading libraries
from pathlib import Path
import geoai
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import numpy as np

In [3]:
## Loading raster data
PROJECT_DIR = Path.cwd().parent
print(PROJECT_DIR.resolve)

raster_dir= PROJECT_DIR / "data" / "raw" / "raster" / "vexcel" / "luzern-2025_epsg2056.tif"

with rasterio.open(raster_dir) as src:
    print("CRS:", src.crs)
    print("Size:", src.width, src.height)
    print("Bands:", src.count)


<bound method Path.resolve of WindowsPath('c:/Users/Maja/Documents/04-Studium/SDS320/project')>
CRS: EPSG:2056
Size: 13334 13334
Bands: 4


In [4]:
# Loading vector data
vector_path = PROJECT_DIR / "data" / "raw" / "vector" / "osm" / "OSM_buildings.geojson"

gdf_buildings = gpd.read_file(vector_path)

print("Vector CRS:", gdf_buildings.crs)
print(gdf_buildings.geom_type.unique())
print(len(gdf_buildings))
print(gdf_buildings.head())

Vector CRS: EPSG:2056
['Polygon']
765
      building                                           geometry
0  residential  POLYGON ((2666198.46 1209935.72, 2666187.977 1...
1   apartments  POLYGON ((2666617 1209968.564, 2666623.853 120...
2   apartments  POLYGON ((2666045.503 1209980.109, 2666036.703...
3          yes  POLYGON ((2666818.372 1209990.739, 2666805.363...
4   apartments  POLYGON ((2666008.969 1209983.273, 2666000.285...


c:\Users\Maja\programming\anaconda3\envs\sds320_v2\Lib\site-packages\pyogrio\core.py:38: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


In [5]:
## Verify correct CRS
if gdf_buildings.crs != src.crs:
    gdf_buildings = gdf_buildings.to_crs(src.crs)

print("Reprojected vector CRS:", gdf_buildings.crs)

Reprojected vector CRS: EPSG:2056


#### B) Building mask  
First, we need to rasterize the building vectors to a building mask (0= no building, 1 = building).

In [6]:
mask_path = PROJECT_DIR / "data" / "processed" / "building_mask.tif"

geoai.vector_to_raster(
    gdf_buildings,
    mask_path,
    reference_raster=raster_dir,
)
# class values should be 1 for buildings and 0 for background
# how can i check this ?????

array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(13334, 13334), dtype=uint8)

Now i try to run several (non-visual) checks to see whether this rasterization actually worked:

In [7]:
# Check unique values of raster mask
with rasterio.open(mask_path) as mask_src:
    mask_data = mask_src.read(1)  # Read the first band
    unique_values = np.unique(mask_data)
    print("Unique values in the raster mask:", unique_values)

Unique values in the raster mask: [0 1]


In [8]:
# count building pixels
with rasterio.open(mask_path) as mask_src:
    mask_data = mask_src.read(1)  # Read the first band
    building_pixel_count = np.sum(mask_data == 1)
    print("Number of building pixels:", building_pixel_count)

Number of building pixels: 50173397


In [9]:
# Calculate percentage coverage of buildings in the raster
with rasterio.open(mask_path) as mask_src:
    mask_data = mask_src.read(1)  # Read the first band
    total_pixels = mask_data.size
    building_pixel_count = np.sum(mask_data == 1)
    building_coverage_percentage = (building_pixel_count / total_pixels) * 100
    print(f"Percentage coverage of buildings in the raster: {building_coverage_percentage:.2f}%")

Percentage coverage of buildings in the raster: 28.22%


In [12]:
# Compapre rasterized area against vector area
vector_area = gdf_buildings.geometry.area.sum()

pixel_size = src.res[0]  # Assuming square pixels, take the x resolution

raster_area = building_coverage_percentage / 100 * total_pixels * pixel_size ** 2  # Assuming pixel_size is the resolution of the raster

print(f"Vector area: {vector_area:.2f} square units")
print(f"Raster area: {raster_area:.2f} square units")
print("Ratio:", raster_area / vector_area)

Vector area: 307044.10 square units
Raster area: 282225.36 square units
Ratio: 0.9191687999084875


Conclusion of all these data checks: Rasterization seems to have worked. There are 0 and 1 as unique values, the percentage covered by buildings seems reasonable and the ratio of vector vs. raster area is also acceptable.  

**What could further be improved and implemented:**
- control for edge cases
- buffers ?
- in general also include more of the checks from the GitBook ? To be done !!!!
- e.g. in the next code cell...

In [17]:
geoai.view_image(mask_path, figsize=(18,10))
# Doesn't work, kernel always crashes....

## how can i control edge cases ???


AttributeError: 'WindowsPath' object has no attribute 'shape'

: 

#### C) Vegetation Mask
Secondly, we create a vegetation mask based on a NDVI threshold.  
-> 0 = no vegetation, 1 = vegetated  

still need to research about the appropriateness of the threshold !!!

In [14]:
## start with a NDVI threshold of 0.2 (starting point)
#  for denser vegetation, we can increase the threshold to 0.3 or 0.4) 
with rasterio.open(raster_dir) as src:
    print(src.count)
    print(src.descriptions)

4
('blue', 'green', 'red', 'nir')


In [15]:
mask_path = PROJECT_DIR / "data" / "processed"

## Compute NDVI
with rasterio.open(raster_dir) as src:

    red = src.read(1).astype(np.float32)   # change band number if needed
    nir = src.read(2).astype(np.float32)   # change band number if needed

    ndvi = (nir - red) / (nir + red + 1e-10)

    profile = src.profile.copy()

## Create the vegetation mask based on the NDVI threshold
ndvi_threshold = 0.2
vegetation_mask = (ndvi >= ndvi_threshold).astype(np.uint8)

## Save the mask
profile.update(
    dtype=rasterio.uint8,
    count=1,
    nodata=0
)

out_path = mask_path / "vegetation_mask.tif"

with rasterio.open(out_path, "w", **profile) as dst:
    dst.write(vegetation_mask, 1)

print(f"Mask saved to: {out_path}")

Mask saved to: c:\Users\Maja\Documents\04-Studium\SDS320\project\data\processed\vegetation_mask.tif


#### D) Raster Calculations
The created building and vegetation masks need to be multiplied and added together.  

Step 1: Multiply building mask with vegetation mask to only get vegetation on buildings and not vegetated area elsewhere  
Step 2: Add output mask from step 1 to original buildings mask.  
Output: mask with code 0 = background; 1 = non-vegetated buildings; 2 = vegetated/green buildings

In [22]:
# Verify alignment of rasters
assert buildings.shape == vegetation_mask.shape, "Rasters are not aligned!"

In [24]:
mask_path = PROJECT_DIR / "data" / "processed"

# Step 1: Multiplication  
with rasterio.open(mask_path / "building_mask.tif") as src:
    buildings = src.read(1)
    profile = src.profile

with rasterio.open(mask_path / "vegetation_mask.tif") as src:
    vegetation = src.read(1)

# Intersection: 1 only where both masks are 1
building_vegetation = buildings * vegetation

print(np.unique(building_vegetation))

## Sanity Check
print("Building values:", np.unique(buildings))
print("Vegetation values:", np.unique(vegetation))
print("Intersection values:", np.unique(building_vegetation))

[0 1]
Building values: [0 1]
Vegetation values: [0 1]
Intersection values: [0 1]


In [18]:
## save intersection mask
with rasterio.open(mask_path / "building_vegetation_mask.tif", "w", **profile) as dst:
    dst.write(building_vegetation, 1)

In [ ]:
# Step 2: Addition (combined = buildings + building_vegetation)
combined = buildings + building_vegetation

print("Final values:", np.unique(combined))

## Check for how amount of pixels per class (percentage)
total = combined.size

for val, count in zip(unique, counts):
    print(
        f"Class {val}: {count:,} pixels "
        f"({100 * count / total:.2f}%)"
    )

Final values: [0 1 2]
Class 0: 127,622,159 pixels (71.78%)
Class 1: 49,700,963 pixels (27.95%)
Class 2: 472,434 pixels (0.27%)


In [21]:
## Save the final raster
with rasterio.open(mask_path / "combined_mask.tif", "w", **profile) as dst:
    dst.write(combined, 1)